In [2]:
import os
from d3rlpy.logging import UnifiedFileAdapterFactory
import numpy as np
os.environ["D3RLPY_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data"
os.makedirs(os.environ["D3RLPY_DATASETS_PATH"], exist_ok=True)
os.environ["MINARI_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data/minari_data"
os.makedirs(os.environ["MINARI_DATASETS_PATH"], exist_ok=True)

import d3rlpy

dataset, env = d3rlpy.datasets.get_minari('atari/pong/expert-v0')
seed = 1

for ep in dataset.episodes:
    print(ep.observations.shape)
    break

dataset_name = "pong"
# fix seed
d3rlpy.seed(seed)
d3rlpy.envs.seed_env(env, seed)
target_return=10
# if "cartpole" in dataset_name:
#     target_return = 200
# else:
#     raise ValueError("unsupported dataset")

discrete_tacr = d3rlpy.algos.DiscreteTACRConfig(
    batch_size=64,
    actor_learning_rate=1e-4,
    actor_optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=10000#10000
        ),
    ),
    actor_encoder_factory=d3rlpy.models.PixelEncoderFactory(
            feature_size=128, exclude_last_activation=True
        ),
    observation_scaler=d3rlpy.preprocessing.PixelObservationScaler(),
    position_encoding_type=d3rlpy.PositionEncodingType.GLOBAL,
    context_size=20,
    num_heads=8,
    num_layers=6,
    max_timestep=200,
    compile_graph=True,
    alpha=0.5,
).create(device="cuda:0")

discrete_tacr.fit(
    dataset,
    n_steps=100000,# 100000,
    n_steps_per_epoch=1000,# 1000,
    save_interval=100,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"Discrete_TACR_{dataset_name}_{seed}",
    logger_adapter=UnifiedFileAdapterFactory(),
    n_trials=50,
    eval_gaps=1
)

2025-07-27 19:48.59 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('uint8')], shape=[(160, 3, 210)]) reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)])
2025-07-27 19:48.59 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-07-27 19:48.59 [info     ] Action size has been automatically determined. action_size=6
(1864, 160, 3, 210)
2025-07-27 19:48.59 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('uint8')], shape=[(160, 3, 210)]), action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]), reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)]), action_space=<ActionSpace.DISCRETE: 2>, action_size=6)
2025-07-27 19:48.59 [debug    ] Building models...            


RuntimeError: Calculated padded input size per channel: (3 x 210). Kernel size: (8 x 8). Kernel size can't be greater than actual input size

In [6]:
type(env.reset())

tuple

In [7]:
one,two = env.reset()

In [8]:
print(type(one))
print(type(two))

<class 'numpy.ndarray'>
<class 'dict'>


In [10]:
print((one.shape))
print((two.keys()))

(210, 160, 3)
dict_keys(['lives', 'episode_frame_number', 'frame_number'])


In [8]:
for ep in dataset.episodes:
    print(type(ep.observations))
    print(type(ep))
    break

<class 'numpy.ndarray'>
<class 'd3rlpy.dataset.components.Episode'>
